# EXP-004 — M3 ASTNN trên Kaggle GPU

**Rẻ nhất trong 5 mô hình: ~35 phút** (DEC-021). Không cần network sau clone, không
cần tokenizer, không truncate.

**Đặc điểm khác M1/M2:** đây là mô hình khác hẳn về họ — GRU 200 đơn vị, **428.001**
tham số (so với 126.416.641 của CodeBERT/GraphCodeBERT), train từ scratch, giữ **toàn
bộ** cây statement. Vì vậy **không** buộc phải giống hyperparameter M1/M2; chỉ giữ
chung seed, số epoch, warmup ratio, early stopping và cách chọn threshold. Khi so
sánh phải nói rõ điểm này, không so số F1 trần (xem DEC-021).

**Setup 1 lần:** Settings → Accelerator = **GPU**, Internet = **ON**.

**Sửa 1 chỗ:** cell 2 — dòng `REPO`. Chạy cell theo thứ tự, **đừng Run All**.

### Nếu muốn train chung phiên với EXP-003
M3 rẻ hơn M2 ~13× nên **chạy notebook này trước** (DEC-023): nếu phiên M2 sau đó chết
giữa chừng, bạn vẫn còn kết quả M3. Tổng cả hai ~8,4h < giới hạn 12h nên vẫn nằm
trong một ngày quota.

### Đánh giá ở đâu?
**Ở Kaggle luôn** (cell 8), tốn ~1 phút. Sau đó chỉ tải metrics + predictions về
(vài MB), khỏi cần tải checkpoint 5MB về chạy lại local.

In [ ]:
import json
import os
import subprocess
import sys

# Cách 1 (khuyên dùng): Kaggle Add-ons → Secrets → GITHUB_TOKEN, attach vào notebook
# Cách 2: dán trực tiếp dưới đây (ĐỪNG share notebook này)
GITHUB_TOKEN = ""  # GitHub PAT — bắt buộc nếu repo PRIVATE
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = GITHUB_TOKEN or UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as e:
    print("dùng token dán tay:", e)
print("GITHUB_TOKEN:", "OK" if GITHUB_TOKEN else "TRỐNG")


In [ ]:
REPO = "kichirou244/SemanticCodeCloneDetection"
assert REPO != "YOUR_USER/YOUR_REPO", "Sửa REPO trước khi chạy cell tiếp theo"

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(f"$ {cmd}\n{r.stdout}{r.stderr}")
    if r.returncode != 0:
        raise RuntimeError(f"lệnh thất bại (rc={r.returncode}): {cmd}")
    return r.stdout

def py(code, **kw):
    """Chạy snippet Python một dòng (không dùng newline trong code)."""
    return sh(f'{sys.executable} -c {json.dumps(code)}', **kw)

# Idempotent: chạy lại cell này sẽ pull bản mới thay vì clone lần nữa.
if os.path.isdir("project"):
    print("project đã tồn tại → pull thay vì clone")
    sh("git -C project pull --ff-only")
else:
    url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git" if GITHUB_TOKEN else f"https://github.com/{REPO}.git"
    sh(f'git clone "{url}" project')
sh(f"git -C project remote set-url origin https://github.com/{REPO}.git")  # xóa token khỏi remote
sh("git -C project log --oneline -1")
os.chdir("project")
print("cwd =", os.getcwd())
leak = GITHUB_TOKEN and GITHUB_TOKEN in sh("git remote -v")
print("còn sót token trong remote?", "CÓ — xóa notebook!" if leak else "KHÔNG")


In [ ]:
# KHÔNG đụng torch: Kaggle đã có bản CUDA tương thích, `pip install transformers`
# vô tình kéo torch về bản CPU/default sẽ làm train rất chậm (hoặc chết).
import importlib

# tree-sitter + tree-sitter-cpp BẮT BUỘC: EXP-004 build cache AST từ parser này.
need = []
for mod, pkg in [("sklearn", "scikit-learn"), ("yaml", "pyyaml"),
                 ("tqdm", "tqdm"), ("numpy", "numpy"),
                 ("tree_sitter", "tree-sitter"), ("tree_sitter_cpp", "tree-sitter-cpp")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)
print("cần cài:", need or "không")
if need:
    sh("pip install -q " + " ".join(need))

import torch, sklearn, numpy
assert torch.cuda.is_available(), "Kaggle chưa gắn GPU — vào Settings → Accelerator → GPU"
print({"torch": torch.__version__, "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0),
       "vram_gb": round(torch.cuda.get_device_properties(0).total_memory/1e9, 1),
       "cpu_cores": os.cpu_count()})
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
# Parser C++ phải nạp được, nếu không thì train sẽ chết ở bước build cache
py("from src.representations.ast.extract import get_parser; t = get_parser('cpp').parse(b'int f(){return 1;}'); print('tree-sitter cpp OK, nodes =', t.root_node.descendant_count)")


In [ ]:
# Rebuild dataset foundation POJ-104 ngay trên Kaggle.
# data/processed + data/splits bị .gitignore nên không có sẵn trong clone — phải build lại.
sh("python scripts/build_dataset.py --config configs/pairs/poj104.yaml")
py("import json; m = json.load(open('data/manifests/poj104.json')); "
   "print('leakage_passed =', m['leakage_passed']); "
   "print('pairs =', {k: v for k, v in m.get('pairs', {}).items()})")
sh("python scripts/env_report.py --output results/exp004_astnn_poj104/env.json")
sh("df -h /kaggle/working | tail -1")
# Cùng sha256 manifest/pairs với EXP-002/003 là điều kiện để so sánh được
sh("sha256sum data/manifests/poj104.json")


In [ ]:
# Preflight: GPU/VRAM/config trước khi tốn 35 phút. Phải IN ra báo cáo.
sh("python -m src.experiments.exp004_astnn --config configs/exp004_astnn.yaml "
   "--mode preflight --device cuda")


In [ ]:
# Diagnostic: xác nhận forward/backward/param thật sự chạy trên GPU (gate của
# reports/010) và in ra số optimizer step đo được để ngoại suy thời gian trên T4.
# Phải in `logit std > 0`; nếu scores hội tụ về 0 thì chưa học được, dừng lại.
sh("python -m src.experiments.exp004_astnn --config configs/exp004_astnn.yaml "
   "--mode diagnostic --device cuda")


In [ ]:
# FULL TRAINING (tieep) — train tu dau, tran 20 epoch (early stopping tu dung).
# Val F1 3 epoch dau con doc (0,643 -> 0,677 -> 0,720) nen cho chay tiep thay vi
# resume: khong phu thuoc file checkpoint cua session cu.
# ~340s/epoch -> tran ~2h + build cache AST; early stopping (patience 2) thuong
# dung som hon nhieu.
# `train` tự gọi evaluate khi hết epoch/hết budget, nhưng cell 8 chạy lại evaluate
# một lần nữa để artifact là nguồn số chuẩn (xem giải thích ở cell 8).
#
# Nếu session bị kill giữa chừng: chép lại last_checkpoint.pt từ version đã save,
# đổi MODE = "resume" rồi chạy lại cell này.
MODE = "train"  # "train" | "resume"
TIME_BUDGET_S = 3 * 3600  # tran 20 epoch + cache; early stopping dung som hon
sh(f"python -m src.experiments.exp004_astnn --config configs/exp004_astnn.yaml "
   f"--mode {MODE} --device cuda --time-budget-s {TIME_BUDGET_S}")


In [ ]:
# EVALUATE — chạy ngay trên Kaggle (~1 phút), KHÔNG tải checkpoint về máy.
#
# Vì sao cần cell này khi `train` đã tự evaluate: để artifact
# (metrics.json, predictions/*.jsonl, EVAL_COMPLETE.json) được sinh bởi MỘT lần
# chạy có thứ tự rõ ràng, và để chạy lại được sau khi tải xuống mà không tốn giờ local.
# Chạy lại nhiều lần là an toàn: ghi đè cùng một thư mục kết quả, không nhân bản.
sh("python -m src.experiments.exp004_astnn --config configs/exp004_astnn.yaml "
   "--mode evaluate --device cuda")
OUT = "results/exp004_astnn_poj104"
sh(f"test -f {OUT}/EVAL_COMPLETE.json && echo 'EVAL COMPLETE' || echo 'EVAL CHUA XONG'")
print(sh(f"cat {OUT}/metrics.json"))


In [ ]:
# Đóng gói kết quả để tải về — CHỈ metrics + predictions (vài MB).
# Checkpoint không tải về: đã evaluate xong ở cell 8, và
# predictions/*.jsonl đủ để tính lại MỌI metric/threshold/error analysis mà không
# cần model (xem reports/011 §11.13).
OUT = "results/exp004_astnn_poj104"
# Đổi sang True chỉ khi muốn giữ `best_weights_only.pt` (~1,7MB) để chạy lại
# evaluate ở máy khác. Mặc định False ⇒ tải về nhẹ nhất.
KEEP_WEIGHTS = False
if KEEP_WEIGHTS:
    py(f"from pathlib import Path; from src.experiments.transformer_pair_runner import export_eval_weights; export_eval_weights(Path('{OUT}'))")
sh(f"tar --exclude='*checkpoint.pt' --exclude='*.tmp' -czf exp004_results.tar.gz {OUT}")
sh("ls -lh exp004_results.tar.gz")
sh(f"ls -lh {OUT}/")
print(sh(f"cat {OUT}/metrics.json"))


### Sau khi tải `exp004_results.tar.gz` về

1. Giải nén **đè** lên `results/exp004_astnn_poj104/`.
2. **Không cần** chạy evaluate local — đã có `metrics.json`, `threshold.json`,
   `confusion_matrix.json`, `model_stats.json` và `predictions/{validation,test}_scores.jsonl`.
   Muốn tính lại số khác (ví dụ F1 ở threshold khác) thì đọc `predictions/` — không cần GPU.
3. Nếu phiên bị kill giữa chừng và bạn còn `last_checkpoint.pt` trong version đã save:
   chép vào thư mục result rồi chạy lại notebook với `MODE = "resume"`.

### Cập nhật docs sau khi có số liệu (repo yêu cầu code + docs tiến song song)

1. `reports/013_astnn_results.md` — test metrics + so sánh M1/M2, nêu rõ M3 **không
   truncate** (giữ trọn cây statement) nên thuận lợi là do **biểu diễn**, không phải do
   "thông minh hơn"; và M3 chỉ 428k tham số nên không so số F1 trần với 126M.
2. `docs/experiment_registry.md` + `configs/exp004_astnn.yaml` — status → `COMPLETED`.
3. `run_metadata.json` phải có `device_actual: cuda:0` + `cuda_device_name` (chứng minh
   GPU thật, gate của DEC-017), và `training_time_s` để so với ước lượng 35 phút.